# Audio preprocessing

This notebook converts the WAV files into Mel-spectrogram images and stores them in `Image_dataset`. Run all cells from top to bottom.

In [ ]:
# Import the libraries used in this notebook.
from pathlib import Path
import shutil
import librosa
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.model_selection import train_test_split


In [ ]:
# Define all project paths.
project_folder = Path.cwd()
data_folder = project_folder / 'Data'
training_audio_folder = data_folder / 'train_audio' / 'train_audio'
testing_audio_folder = data_folder / 'test_audio' / 'test_audio'
image_dataset_folder = project_folder / 'Image_dataset'

# Audio settings.
sample_rate = 22050
audio_duration_seconds = 5
expected_audio_samples = sample_rate * audio_duration_seconds
number_of_mel_bands = 128
fft_size = 1024
hop_length = 512
image_size = (128, 128)
random_seed = 42

training_table = pd.read_csv(data_folder / 'train.csv')
testing_table = pd.read_csv(data_folder / 'test.csv')

print('Training files:', len(training_table))
print('Testing files:', len(testing_table))
print('Number of classes:', training_table['target'].nunique())


In [ ]:
# Check that the CSV files point to real audio files.
for filename in training_table['filename']:
    audio_path = training_audio_folder / filename
    if not audio_path.exists():
        raise FileNotFoundError(f'Missing training file: {audio_path}')

for filename in testing_table['filename']:
    audio_path = testing_audio_folder / filename
    if not audio_path.exists():
        raise FileNotFoundError(f'Missing testing file: {audio_path}')

# Split original files before creating images.
# This prevents related data from appearing in both sets.
training_file_table, validation_file_table = train_test_split(
    training_table,
    test_size=0.20,
    random_state=random_seed,
    stratify=training_table['target']
)
print('Training files:', len(training_file_table))
print('Validation files:', len(validation_file_table))


In [ ]:
def load_audio(audio_path):
    # Load every recording as mono audio at the same sample rate.
    audio_signal, _ = librosa.load(audio_path, sr=sample_rate, mono=True)

    # Pad short recordings with zeros.
    if len(audio_signal) < expected_audio_samples:
        missing_samples = expected_audio_samples - len(audio_signal)
        audio_signal = np.pad(audio_signal, (0, missing_samples))

    # Cut long recordings to exactly five seconds.
    if len(audio_signal) > expected_audio_samples:
        audio_signal = audio_signal[:expected_audio_samples]

    return audio_signal

def audio_to_spectrogram_image(audio_path):
    # Convert the audio waveform into a Mel-spectrogram.
    audio_signal = load_audio(audio_path)
    mel_spectrogram = librosa.feature.melspectrogram(
        y=audio_signal,
        sr=sample_rate,
        n_fft=fft_size,
        hop_length=hop_length,
        n_mels=number_of_mel_bands,
        power=2.0
    )

    # Decibel values are easier for the CNN to learn.
    log_mel_spectrogram = librosa.power_to_db(mel_spectrogram, ref=np.max)

    # Scale the values to the valid grayscale image range 0 to 255.
    minimum_value = log_mel_spectrogram.min()
    maximum_value = log_mel_spectrogram.max()
    value_range = maximum_value - minimum_value
    if value_range == 0:
        value_range = 1
    image_values = (log_mel_spectrogram - minimum_value) / value_range * 255
    image_values = image_values.astype(np.uint8)

    # Resize all images to the same dimensions.
    image = Image.fromarray(image_values, mode='L')
    image = image.resize(image_size)
    return image


In [ ]:
def save_labelled_images(file_table, audio_folder, split_name):
    # Create folders such as Image_dataset/train/class_00.
    split_folder = image_dataset_folder / split_name
    number_of_files = len(file_table)

    for file_number, row in enumerate(file_table.itertuples(index=False), start=1):
        target_number = int(row.target)
        class_folder = split_folder / f'class_{target_number:02d}'
        class_folder.mkdir(parents=True, exist_ok=True)

        audio_path = audio_folder / row.filename
        image_name = Path(row.filename).stem + '.png'
        image_path = class_folder / image_name
        image = audio_to_spectrogram_image(audio_path)
        image.save(image_path)

        if file_number % 100 == 0 or file_number == number_of_files:
            print(f'{split_name}: {file_number} of {number_of_files} files')

def save_test_images(file_table):
    # Test labels are not available, so test images go into one folder.
    test_folder = image_dataset_folder / 'test' / 'unlabelled'
    test_folder.mkdir(parents=True, exist_ok=True)

    for file_number, row in enumerate(file_table.itertuples(index=False), start=1):
        audio_path = testing_audio_folder / row.filename
        image_name = Path(row.filename).stem + '.png'
        image_path = test_folder / image_name
        image = audio_to_spectrogram_image(audio_path)
        image.save(image_path)

        if file_number % 100 == 0 or file_number == len(file_table):
            print(f'test: {file_number} of {len(file_table)} files')


In [ ]:
# Delete only the previous generated image dataset.
if image_dataset_folder.exists():
    shutil.rmtree(image_dataset_folder)

# Create training, validation, and test images.
save_labelled_images(training_file_table, training_audio_folder, 'train')
save_labelled_images(validation_file_table, training_audio_folder, 'validation')
save_test_images(testing_table)

print('Finished. Images are stored in:', image_dataset_folder)


In [ ]:
# Display one generated image for inspection.
example_image_path = next((image_dataset_folder / 'train').glob('class_*/*.png'))
example_image = Image.open(example_image_path)
print('Example image:', example_image_path)
display(example_image.resize((384, 384)))
